In [0]:
from pyspark.sql import functions as F

CATALOG = dbutils.widgets.get("catalog")
RAW_SCHEMA = dbutils.widgets.get("stream_schema")
TITLE = dbutils.widgets.get("title")
BRONZE_SCHEMA = f"{RAW_SCHEMA}_bronze"
SILVER_SCHEMA = f"{RAW_SCHEMA}_silver"


BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{TITLE}"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TITLE}"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

In [0]:
bronze_stream_df = spark.readStream.table(BRONZE_TABLE)

In [0]:
cleaned_df = (
    bronze_stream_df
        .withColumn("show_id", F.trim("show_id"))
        .withColumn("type", F.initcap(F.trim("type")))
        .withColumn(
            "type",
            F.when(F.lower(F.trim(F.col("type"))) == "movie", "Movie")
            .when(F.lower(F.trim(F.col("type"))) == "tv show", "TV Show")
            .otherwise(F.trim(F.col("type")))
        )
        .withColumn("title", F.trim("title"))
        .withColumn("director", F.trim("director"))
        .withColumn("country", F.trim("country"))
        .withColumn("rating", F.upper(F.trim("rating")))
        .withColumn("date_added",  F.try_to_date(F.trim("date_added"), F.lit("MMMM d, yyyy")))
        .withColumn("release_year", F.col("release_year").cast("int"))
        .withColumn("duration", F.trim("duration"))
        .withColumn("listed_in", F.trim("listed_in"))
        .withColumn("description", F.trim("description"))
)

In [0]:
silver_df = (
    cleaned_df
    .withColumn(
        "audience_category",
        F.when(F.col("rating").isin("TV-Y", "TV-Y7", "G"), "Children")
         .when(F.col("rating").isin("TV-G", "PG", "TV-PG"), "Family")
         .when(F.col("rating").isin("PG-13", "TV-14"), "Teen")
         .when(F.col("rating").isin("R", "NC-17", "TV-MA"), "Adults")
         .otherwise("Unknown")
    )
    .withColumn(
        "release_period",
        F.when(F.col("release_year") < 2000, "Before 2000")
         .when(F.col("release_year") < 2010, "2000-2009")
         .when(F.col("release_year") < 2020, "2010-2019")
         .otherwise("2020+")
    )
    .withColumn("has_director", F.col("director").isNotNull())
    .withColumn("silver_created_at", F.current_timestamp())
    .withColumn("silver_updated_at", F.current_timestamp())
)